In [1]:
print("FLAN-T5 training notebook started")

FLAN-T5 training notebook started


In [1]:
import pandas as pd

from datasets import Dataset

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer
)

In [2]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/validation.csv")
test_df = pd.read_csv("../data/test.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(551, 3)
(69, 3)
(69, 3)


In [3]:
train_df["input_text"] = "question: " + train_df["Question"].astype(str)
train_df["target_text"] = train_df["Answer"].astype(str)

val_df["input_text"] = "question: " + val_df["Question"].astype(str)
val_df["target_text"] = val_df["Answer"].astype(str)

test_df["input_text"] = "question: " + test_df["Question"].astype(str)
test_df["target_text"] = test_df["Answer"].astype(str)

train_df[["input_text", "target_text"]].head()

,input_text,target_text
0,question: what does memantine look like,"Color - ORANGE, Shape - CAPSULE (biconvex), Sc..."
1,question: how does a 20 mcg bedford norton tra...,25 mcg fentanyl/hr = 40 mg oral / 20 mg IV oxy...
2,question: what mg norco comes in,... NORCO® 5/325 ... NORCO® 7.5/325 ... NORCO®...
3,question: vyvanse 10 what is all in this pill ...,The following adverse reactions are discussed ...
4,question: dtap/tdap/td vaccines how often is t...,"Routine Vaccination of Infants and Children, A..."


In [4]:
train_dataset = Dataset.from_pandas(
    train_df[["input_text", "target_text"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["input_text", "target_text"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["input_text", "target_text"]]
)

print(train_dataset)

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 551
})


In [5]:
model_name = "google/flan-t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)

model = T5ForConditionalGeneration.from_pretrained(model_name)

print("FLAN-T5 model loaded successfully")

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\Serhat\Desktop\medicine-chatbot-nlp\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Serhat\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded successfully


In [6]:
def preprocess_function(examples):

    model_inputs = tokenizer(
        examples["input_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["target_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [7]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True
)

print("Tokenization completed")

Map:   0%|          | 0/551 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Tokenization completed


In [8]:
training_args = TrainingArguments(
    output_dir="../models/flan-t5-results",

    eval_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=1,

    weight_decay=0.01,

    save_total_limit=1,

    logging_steps=50
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

print("Trainer created successfully")

Trainer created successfully


In [10]:
trainer.train()


c:\Users\Serhat\Desktop\medicine-chatbot-nlp\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,14.525530,12.350507


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Serhat\Desktop\medicine-chatbot-nlp\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=138, training_loss=15.71274069081182, metrics={'train_runtime': 101.7534, 'train_samples_per_second': 5.415, 'train_steps_per_second': 1.356, 'total_flos': 25606417022976.0, 'train_loss': 15.71274069081182, 'epoch': 1.0})

In [11]:
model.save_pretrained("../models/flan-t5-small-medicationqa-final")
tokenizer.save_pretrained("../models/flan-t5-small-medicationqa-final")

print("FLAN-T5 model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FLAN-T5 model saved successfully.
